This notebook runs the training for finetuning dinov2 on the image dataset

Please modify the `run` variable and the model [configuration](#configuration)

# Setup

In [1]:
import os

from tensorflow.keras.utils import image_dataset_from_directory
from tensorflow.data import AUTOTUNE
from tensorflow.config import list_physical_devices

os.environ["KERAS_BACKEND"] = "tensorflow" # Making sure keras backend is the right one
import keras
import keras_hub

In [2]:
# Try to use requirements.txt and fallback on the full pip command.
!pip install -r ../requirements.txt || pip install pytest pylint ipdb jupyterlab numpy pandas matplotlib seaborn scikit-learn tensorflow timm transformers keras_hub==0.26.0

ERROR: Could not open requirements file: [Errno 2] No such file or directory: '../requirements.txt'


In [3]:
# How do you want to run it?
run = "colab"
# run = "local"

In [4]:
if run == "colab":
    # Connect to google drive
    from google.colab import drive
    drive.mount('/content/drive')

    # Set the default root path of Kinoko project
    ROOT = "/content/drive/MyDrive/Colab Notebooks/lewagon/Kinoko"

    print(list_physical_devices('GPU'))
elif run == "local":
    ROOT = "../"
else:
    print("Error: variable `run` is set as an unknown type")


Mounted at /content/drive
[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [5]:
# Setting image dataset directory
image_data_dir = f"{ROOT}/data/image_dataset"
image_dataset = image_dataset_from_directory(image_data_dir,
                                             labels="inferred",
                                             label_mode="binary",
                                             )

Found 10447 files belonging to 2 classes.


# Modeling

## Configuration

Modify this if you need

In [6]:
IMG_SIZE = 518 # DINOv2 Base preset expects 518x518
BATCH_SIZE = 32
EPOCHS = 50
SEED = 42

## Data preparation

In [7]:
# train set is 70%
# val set is 15%
# test set is 15%

train_ds = image_dataset_from_directory(
    image_data_dir,
    labels="inferred",
    label_mode="binary",
    validation_split=0.3,
    subset="training",
    seed=SEED,
    image_size=(IMG_SIZE, IMG_SIZE), # Changed to 518
    batch_size=BATCH_SIZE
)

test_val_ds = image_dataset_from_directory(
    image_data_dir,
    labels="inferred",
    label_mode="binary",
    validation_split=0.3,
    subset="validation",
    seed=SEED,
    image_size=(IMG_SIZE, IMG_SIZE), # Changed to 518
    batch_size=BATCH_SIZE
)

# Split test and val data
half_test_val_size = int(len(test_val_ds)/2)
test_ds = test_val_ds.take(half_test_val_size)
val_ds = test_val_ds.skip(half_test_val_size)

# Prefetching for faster loading in GPU
train_ds.prefetch(AUTOTUNE)
test_ds.prefetch(AUTOTUNE)
val_ds.prefetch(AUTOTUNE)

Found 10447 files belonging to 2 classes.
Using 7313 files for training.
Found 10447 files belonging to 2 classes.
Using 3134 files for validation.


<_PrefetchDataset element_spec=(TensorSpec(shape=(None, 518, 518, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None, 1), dtype=tf.float32, name=None))>

## Model loading

In [ ]:
backbone = keras_hub.models.DINOV2Backbone.from_preset("dinov2_base")

# Freeze the backbone
backbone.trainable = False

# Build the model
inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))

# ImageNet Normalization
x = keras.layers.Rescaling(1/255.0)(inputs)
x = keras.layers.Normalization(
    mean=[0.485, 0.456, 0.406],
    variance=[0.229**2, 0.224**2, 0.225**2]
)(x)

# Pass to backbone
backbone_out = backbone({"images": x})  # convert to dict
outputs = backbone_out[:, 0, :]  # Get only CLS token

## New head

In [ ]:
x = keras.layers.Dense(512, activation="relu")(outputs)
x = keras.layers.Dropout(0.3)(x)
predictions = keras.layers.Dense(1, activation="sigmoid")(x)

model = keras.Model(inputs=inputs, outputs=predictions)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss="binary_crossentropy",
    metrics=["accuracy", "recall", "precision"]
)

model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 518, 518, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling_1 (Rescaling)         │ (None, 518, 518, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ normalization_1 (Normalization) │ (None, 518, 518, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dinov2_backbone                 │ (None, 1370, 768)      │    87,632,640 │
│ (DINOV2Backbone)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ get_item_1 (GetItem)            │ (None, 768)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 512)            │       393,728 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_25 (Dropout)            │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │           513 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 88,026,881 (335.80 MB)

 Trainable params: 394,241 (1.50 MB)

 Non-trainable params: 87,632,640 (334.29 MB)

# Training

## Callbacks

In [ ]:
# --- CALLBACKS ---
callbacks = [
    # Stop early if val_loss stops improving
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    ),

    # Reduce LR on plateau (helps squeeze out last gains)
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-7
    ),

    # Saving model if it's better than the one saved before
    keras.callbacks.ModelCheckpoint(
      filepath=f"{ROOT}/checkpoints/dinov2.keras",
      save_best_only=True,
      save_freq="epoch",  # every N epochs, or "epoch" for every one
    ),
]

## Training

In [ ]:
print("\nStarting training...")
model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks
)


Starting training...
Epoch 1/50


KeyboardInterrupt: 

## Full fine tuning (optional) (not tested)

In [ ]:
# # --- 5. OPTIONAL: FULL FINE-TUNING ---
# # After the head is trained, you can unfreeze and train with a tiny learning rate
# print("\nUnfreezing backbone for fine-tuning...")
# backbone.trainable = True
# model.compile(
#     optimizer=keras.optimizers.Adam(learning_rate=1e-6), # Extremely low LR
#     loss="categorical_crossentropy",
#     metrics=["accuracy"]
# )
# # model.fit(train_ds, validation_data=val_ds, epochs=2)

# # --- 6. EVALUATION & PREDICTION ---
# print("\nEvaluating model...")
# loss, accuracy = model.evaluate(val_ds)
# print(f"Validation Accuracy: {accuracy*100:.2f}%")

# # Prediction example
# sample_img = np.random.rand(1, IMG_SIZE, IMG_SIZE, 3).astype("float32")
# prediction = model.predict(sample_img)
# print(f"Prediction shape: {prediction.shape}")

## Test

### Load model

In [8]:
model_path = f"{ROOT}/checkpoints/dinov2_epoch_06.keras"

model = keras.models.load_model(model_path)

In [9]:
results = model.evaluate(test_ds)

49/49 ━━━━━━━━━━━━━━━━━━━━ 289s 5s/step - accuracy: 0.9114 - loss: 0.2278 - precision: 0.8767 - recall: 0.9729


In [18]:
print(f"loss: {results[0]}\naccuracy: {results[1]}\nrecall: {results[2]}\nprecision: {results[3]}")

loss: 0.22779890894889832
accuracy: 0.9113520383834839
recall: 0.9728773832321167
precision: 0.8767268657684326
